<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Bernoulli_Equation_Pingpong_Ball.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computational Fluid Dynamics Simulation: Bernoulli's Principle

## Project Overview
This notebook contains a scientifically accurate numerical simulation and high-contrast vertical animation (9:16) of a ping-pong ball suspended in an air jet. The simulation demonstrates the physical forces that create a stable aerodynamic trap, allowing the ball to remain centered even when subjected to external disturbances.

## Theoretical Background

### Bernoulli's Principle
Bernoulli's principle states that for an inviscid flow of a non-conducting fluid, an increase in the speed of the fluid occurs simultaneously with a decrease in static pressure or a decrease in the fluid's potential energy. In this simulation, the air jet has its highest velocity at the central axis. According to Bernoulli's equation, this high-velocity core is a region of lower pressure compared to the slower-moving air at the edges of the jet.

### The Coanda Effect and Pressure Gradient Force
The Coanda effect describes the tendency of a fluid jet to stay attached to a convex surface. As air flows around the curved surface of the ping-pong ball, the pressure drops. If the ball moves away from the center of the jet, it encounters higher pressure on the outer side and lower pressure on the side closer to the high-velocity core. This pressure difference generates a net restoring force (a pressure gradient force) that pushes the ball back toward the center of the stream.

### Stability and Equilibrium
The vertical position of the ball is determined by the equilibrium between gravity (downward), the buoyant force (upward, though negligible here), and the aerodynamic drag force (upward). The lateral stability is maintained by the 'stiff' aerodynamic trap created by the pressure gradient. In this model, we utilize a 2D Gaussian profile for the jet velocity and implement horizontal damping to simulate real-world turbulent energy dissipation.

## Simulation Features
- **Physics Engine:** Built with NumPy, using sub-stepping for numerical integration stability.
- **Visualization:** Matplotlib-based animation with dynamic pressure field heatmaps (cyan/red) and particle tracers to visualize flow direction.
- **Readouts:** Live telemetry including ball speed, core air speed, net force in millinewtons (mN), and lateral offset in millimeters (mm).
- **Stability Testing:** The simulation includes programmed force disturbances to demonstrate the system's ability to self-correct and maintain equilibrium.


In [38]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Bernoulli's Equation
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle, FancyBboxPatch, Rectangle, FancyArrowPatch
from IPython.display import display, Video
from google.colab import files

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
CONFIG = {
    "width": 720,
    "height": 1280,
    "fps": 30,
    "main_duration": 33.0,
    "closing_duration": 2.0,
    "ball_radius": 0.02,
    "ball_mass": 0.0027,
    "rho_air": 1.225,
    "gravity": 9.81,
    "drag_coefficient": 0.47,
    "jet_v0": 18.0,
    "jet_decay": 0.18,
    "nozzle_r": 0.012,
    "plume_spread": 0.18,
    "floor_y": 0.03,
    "ceiling_y": 0.28,
    "start_y": 0.12,
    "k0": 0.55,
    "k1": 1.20,
    "k3": 8.00,
    "c_lat": 0.08,
    "disturb_start": 13.0,
    "disturb_end": 13.6,
    "disturb_2_start": 23.0,
    "disturb_2_end": 23.6,
    "disturbance_force": 0.006,
    "substeps": 4,
    "xlim": (-0.08, 0.08),
    "ylim": (0.00, 0.22),
    "field_nx": 120,
    "field_ny": 220,
    "tracer_count": 90,
    "tracer_speed_scale": 1.00,
    "bg_color": "#0A0F1E",
    "panel_color": "#10172B",
    "panel_edge": "#24324F",
    "cyan": "#4DEBFF",
    "blue": "#2C7DFF",
    "orange": "#FF9F43",
    "gold": "#FFD166",
    "white": "#F4F7FB",
    "muted": "#B9C2D3",
    "watermark": "© Mugambi Ndwiga / @craftsandengineering",
    "video_path": "bernoulli_ping_pong_stability.mp4",
    "bitrate": 900,
}

# -------------------------------------------------------------------
# Utilities
# -------------------------------------------------------------------
def phase_text(t, cfg):
    if t < 4:
        return "Why does it stay up?", "The jet is not just lifting the ball. It is stabilizing it."
    if t < 10:
        return "Fast air in the core", "Higher speed in the center means lower pressure there."
    if t < 18:
        return "A stable balance point", "The ball settles where lift, drag, and gravity nearly balance."
    if t < 25:
        return "A safe disturbance", "The nudge is kept within the restoring range."
    if t < 31:
        return "Bernoulli's relation", "P + 1/2 ρv² + ρgz ≈ constant along a streamline."
    return "Not magic. Fluid dynamics.", "The final frame holds the idea in one clean sentence."

def jet_speed(x, y, cfg):
    sigma = cfg["nozzle_r"] + cfg["plume_spread"] * np.maximum(y, 0.0)
    return cfg["jet_v0"] * np.exp(-y / cfg["jet_decay"]) * np.exp(-(x * x) / (2.0 * sigma * sigma + 1e-12))

def visual_flow_field(grid_x, grid_y, ball_x, ball_y, cfg):
    base_v = jet_speed(grid_x, grid_y, cfg)

    dx = grid_x - ball_x
    dy = grid_y - ball_y
    r2 = dx * dx + dy * dy
    r = np.sqrt(r2 + 1e-12)

    obstacle_sigma = 1.8 * cfg["ball_radius"]
    obstacle = np.exp(-r2 / (2.0 * obstacle_sigma * obstacle_sigma + 1e-12))

    u = 0.42 * base_v * obstacle * (-dx / (r + 1e-12))
    v = base_v * (1.0 - 0.38 * obstacle) + 0.05 * base_v * np.tanh(-(dy) / (cfg["ball_radius"] * 1.8))

    u *= 0.18
    v = np.clip(v, 0.0, None)

    pressure = -0.5 * cfg["rho_air"] * (u * u + v * v)
    return u, v, pressure

def sample_flow(x, y, ball_x, ball_y, cfg):
    sigma = cfg["nozzle_r"] + cfg["plume_spread"] * np.maximum(y, 0.0)
    base_v = cfg["jet_v0"] * np.exp(-y / cfg["jet_decay"]) * np.exp(-(x * x) / (2.0 * sigma * sigma + 1e-12))

    dx = x - ball_x
    dy = y - ball_y
    r2 = dx * dx + dy * dy
    obstacle_sigma = 1.8 * cfg["ball_radius"]
    obstacle = np.exp(-r2 / (2.0 * obstacle_sigma * obstacle_sigma + 1e-12))

    ux = 0.16 * base_v * obstacle * (-dx / (np.sqrt(r2 + 1e-12)))
    uy = base_v * (1.0 - 0.30 * obstacle) + 0.03 * base_v * np.tanh(-(dy) / (cfg["ball_radius"] * 2.0))
    uy = max(0.0, uy)
    return ux, uy, -0.5 * cfg["rho_air"] * (ux * ux + uy * uy)

def simulate_ball(cfg):
    fps = cfg["fps"]
    dt = 1.0 / fps
    steps = int(round(cfg["main_duration"] * fps))
    substeps = int(cfg["substeps"])
    h = dt / substeps

    x = 0.0
    y = cfg["start_y"]
    vx = 0.0
    vy = 0.0

    A = np.pi * cfg["ball_radius"] ** 2
    m = cfg["ball_mass"]
    g = cfg["gravity"]
    rho = cfg["rho_air"]
    Cd = cfg["drag_coefficient"]

    hist = {
        "x": np.zeros(steps),
        "y": np.zeros(steps),
        "vx": np.zeros(steps),
        "vy": np.zeros(steps),
        "Vcore": np.zeros(steps),
        "Fx": np.zeros(steps),
        "Fy": np.zeros(steps),
        "disturb": np.zeros(steps),
    }

    for i in range(steps):
        t = i * dt
        for s in range(substeps):
            ts = t + s * h

            disturbance = 0.0
            if cfg["disturb_start"] <= ts <= cfg["disturb_end"]:
                disturbance = cfg["disturbance_force"]
            elif cfg["disturb_2_start"] <= ts <= cfg["disturb_2_end"]:
                disturbance = -cfg["disturbance_force"] # Reversed direction

            V = jet_speed(x, y, cfg)
            Vrel = V - vy

            Fy = 0.5 * rho * Cd * A * Vrel * abs(Vrel) - m * g

            k_lat = cfg["k0"] + cfg["k1"] * (V / cfg["jet_v0"]) ** 2
            Fx = -(k_lat * x + cfg["k3"] * x ** 3) - cfg["c_lat"] * vx + disturbance

            ax = Fx / m
            ay = Fy / m

            vx += ax * h
            vy += ay * h
            x += vx * h
            y += vy * h

            if y < cfg["floor_y"]:
                y = cfg["floor_y"]
                vy = max(vy, 0.0) * 0.2
            if y > cfg["ceiling_y"]:
                y = cfg["ceiling_y"]
                vy = min(vy, 0.0) * -0.15

        hist["x"][i] = x
        hist["y"][i] = y
        hist["vx"][i] = vx
        hist["vy"][i] = vy
        hist["Vcore"][i] = jet_speed(x, y, cfg)
        hist["Fx"][i] = Fx
        hist["Fy"][i] = Fy
        hist["disturb"][i] = disturbance

    return hist

def make_initial_tracers(cfg):
    n = cfg["tracer_count"]
    xs = np.random.uniform(cfg["xlim"][0] * 0.55, cfg["xlim"][1] * 0.55, size=n)
    ys = np.random.uniform(cfg["floor_y"] + 0.005, cfg["floor_y"] + 0.11, size=n)
    return np.column_stack([xs, ys])

def update_tracers(tracers, ball_x, ball_y, cfg, dt):
    for i in range(len(tracers)):
        x, y = tracers[i]
        ux, uy, _ = sample_flow(x, y, ball_x, ball_y, cfg)
        tracers[i, 0] += ux * dt * cfg["tracer_speed_scale"]
        tracers[i, 1] += uy * dt * cfg["tracer_speed_scale"]

        if tracers[i, 1] > cfg["ylim"][1] * 0.98:
            tracers[i, 0] = np.random.uniform(cfg["xlim"][0] * 0.55, cfg["xlim"][1] * 0.55)
            tracers[i, 1] = cfg["floor_y"] + np.random.uniform(0.0, 0.02)
        if tracers[i, 0] < cfg["xlim"][0] or tracers[i, 0] > cfg["xlim"][1]:
            tracers[i, 0] = np.clip(tracers[i, 0], cfg["xlim"][0] * 0.9, cfg["xlim"][1] * 0.9)

    return tracers

def render_video(hist, cfg):
    total_main_frames = len(hist["x"])
    closing_frames = int(round(cfg["closing_duration"] * cfg["fps"]))
    total_frames = total_main_frames + closing_frames
    dt = 1.0 / cfg["fps"]

    fig = plt.figure(figsize=(9, 16), dpi=100, facecolor=cfg["bg_color"])

    main_ax = fig.add_axes([0.05, 0.22, 0.90, 0.70], facecolor=cfg["bg_color"])
    panel_ax = fig.add_axes([0.04, 0.03, 0.92, 0.14], facecolor="none")

    for ax in (main_ax, panel_ax):
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    main_ax.set_xlim(cfg["xlim"])
    main_ax.set_ylim(cfg["ylim"])
    main_ax.set_aspect("equal", adjustable="box")

    gx = np.linspace(cfg["xlim"][0], cfg["xlim"][1], cfg["field_nx"])
    gy = np.linspace(cfg["ylim"][0], cfg["ylim"][1], cfg["field_ny"])
    GX, GY = np.meshgrid(gx, gy)

    init_x = hist["x"][0]
    init_y = hist["y"][0]
    U, V, P = visual_flow_field(GX, GY, init_x, init_y, cfg)

    pmin = -0.5 * cfg["rho_air"] * cfg["jet_v0"] ** 2
    pmax = 0.0

    img = main_ax.imshow(
        P,
        extent=[cfg["xlim"][0], cfg["xlim"][1], cfg["ylim"][0], cfg["ylim"][1]],
        origin="lower",
        cmap="RdYlBu_r",
        vmin=pmin,
        vmax=pmax,
        alpha=0.62,
        interpolation="bilinear",
        zorder=0
    )

    shadow = Circle((0.0, 0.0), cfg["ball_radius"] * 0.90, facecolor="#000000", edgecolor="none", alpha=0.18, zorder=4)
    main_ax.add_patch(shadow)

    ball = Circle((init_x, init_y), cfg["ball_radius"], facecolor="#F7F7F4", edgecolor="#D7DDE7", linewidth=1.2, zorder=6)
    main_ax.add_patch(ball)

    highlight = Circle((init_x - 0.006, init_y + 0.006), cfg["ball_radius"] * 0.28, facecolor="#FFFFFF", edgecolor="none", alpha=0.45, zorder=7)
    main_ax.add_patch(highlight)

    tracers = make_initial_tracers(cfg)
    tracer_scatter = main_ax.scatter(
        tracers[:, 0],
        tracers[:, 1],
        s=14,
        c=cfg["cyan"],
        alpha=0.45,
        linewidths=0,
        zorder=2
    )

    disturb_arrow = FancyArrowPatch(
        posA=(-0.055, init_y + 0.020),
        posB=(-0.025, init_y + 0.020),
        arrowstyle="-|>",
        mutation_scale=20,
        linewidth=2.2,
        color=cfg["orange"],
        alpha=0.0,
        zorder=8
    )
    main_ax.add_patch(disturb_arrow)

    title = fig.text(
        0.5, 0.955,
        "Floating Ping Pong Ball in a Jet Stream",
        ha="center",
        va="top",
        color=cfg["white"],
        fontsize=23,
        weight="bold"
    )
    subtitle = fig.text(
        0.5, 0.925,
        "A numerically driven stability demonstration",
        ha="center",
        va="top",
        color=cfg["muted"],
        fontsize=12
    )
    watermark = fig.text(
        0.965, 0.018,
        cfg["watermark"],
        ha="right",
        va="bottom",
        color=cfg["white"],
        fontsize=9,
        alpha=0.60
    )

    panel_card = FancyBboxPatch(
        (0.00, 0.00), 1.00, 1.00,
        boxstyle="round,pad=0.02,rounding_size=0.03",
        facecolor=cfg["panel_color"],
        edgecolor=cfg["panel_edge"],
        linewidth=1.2,
        transform=panel_ax.transAxes,
        zorder=1
    )
    panel_ax.add_patch(panel_card)

    panel_title = panel_ax.text(
        0.03, 0.80, "",
        transform=panel_ax.transAxes,
        ha="left",
        va="top",
        fontsize=18,
        color=cfg["white"],
        weight="bold",
        zorder=2
    )
    panel_body = panel_ax.text(
        0.03, 0.42, "",
        transform=panel_ax.transAxes,
        ha="left",
        va="top",
        fontsize=14,
        color=cfg["muted"],
        zorder=2
    )
    panel_rhs = panel_ax.text(
        0.03, 0.14, "",
        transform=panel_ax.transAxes,
        ha="left",
        va="top",
        fontsize=12,
        color=cfg["muted"],
        zorder=2
    )

    metric_card = FancyBboxPatch(
        (0.72, 0.16), 0.25, 0.70,
        boxstyle="round,pad=0.02,rounding_size=0.03",
        facecolor="#0D1324",
        edgecolor="#25314A",
        linewidth=1.0,
        transform=panel_ax.transAxes,
        zorder=2
    )
    panel_ax.add_patch(metric_card)

    metric_title = panel_ax.text(
        0.745, 0.77, "Live readout",
        transform=panel_ax.transAxes,
        fontsize=12,
        color=cfg["white"],
        weight="bold",
        ha="left",
        va="top",
        zorder=3
    )
    metric_text = panel_ax.text(
        0.745, 0.57, "",
        transform=panel_ax.transAxes,
        fontsize=11,
        color=cfg["muted"],
        ha="left",
        va="top",
        linespacing=1.35,
        zorder=3
    )

    closing_title = main_ax.text(
        0.5, 0.60,
        "Made by Mugambi Ndwiga",
        transform=main_ax.transAxes,
        ha="center",
        va="center",
        color=cfg["white"],
        fontsize=24,
        weight="bold",
        zorder=10,
        visible=False
    )
    closing_subtitle = main_ax.text(
        0.5, 0.50,
        "@craftsandengineering",
        transform=main_ax.transAxes,
        ha="center",
        va="center",
        color=cfg["white"],
        fontsize=18,
        zorder=10,
        visible=False
    )

    def update(frame):
        nonlocal tracers

        if frame < total_main_frames:
            t = frame * dt
            x = hist["x"][frame]
            y = hist["y"][frame]
            vx = hist["vx"][frame]
            vy = hist["vy"][frame]
            Vcore = hist["Vcore"][frame]
            Fx = hist["Fx"][frame]
            Fy = hist["Fy"][frame]

            U, V, P = visual_flow_field(GX, GY, x, y, cfg)
            img.set_data(P)

            tracers = update_tracers(tracers, x, y, cfg, dt)
            tracer_scatter.set_offsets(tracers)

            ball.center = (x, y)
            shadow.center = (x + 0.002, y - 0.004)
            highlight.center = (x - 0.006, y + 0.006)

            if cfg["disturb_start"] <= t <= cfg["disturb_end"]:
                disturb_arrow.set_alpha(0.95)
                disturb_arrow.set_positions((x - 0.050, y + 0.020), (x - 0.020, y + 0.020))
            elif cfg["disturb_2_start"] <= t <= cfg["disturb_2_end"]:
                disturb_arrow.set_alpha(0.95)
                disturb_arrow.set_positions((x + 0.050, y + 0.020), (x + 0.020, y + 0.020))
            else:
                disturb_arrow.set_alpha(0.0)

            title_text, body_text = phase_text(t, cfg)
            panel_title.set_text(title_text)
            panel_body.set_text(body_text)

            speed = float(np.hypot(vx, vy))
            net_force = float(np.hypot(Fx, Fy))
            stability_note = "Centered" if abs(x) < 0.003 else "Recovering"
            metric_text.set_text(
                f"Ball speed: {speed:0.2f} m/s\n"
                f"Core air speed: {Vcore:0.2f} m/s\n"
                f"Net force: {net_force*1000:0.2f} mN\n"
                f"Lateral offset: {x*1000:0.1f} mm\n"
                f"State: {stability_note}"
            )

            if 25.0 <= t < 31.0:
                panel_rhs.set_text("P + 1/2 ρv² + ρgz ≈ constant")
            else:
                panel_rhs.set_text("")

        else:
            main_ax.set_facecolor(cfg["bg_color"])
            img.set_alpha(0.0)
            tracer_scatter.set_alpha(0.0)
            ball.set_alpha(0.0)
            shadow.set_alpha(0.0)
            highlight.set_alpha(0.0)
            disturb_arrow.set_alpha(0.0)
            panel_ax.set_visible(False)
            title.set_text("")
            subtitle.set_text("")
            watermark.set_text("")
            closing_title.set_visible(True)
            closing_subtitle.set_visible(True)

        return []

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=total_frames,
        interval=1000 / cfg["fps"],
        blit=False
    )

    x_min = float(np.min(hist["x"]))
    x_max = float(np.max(hist["x"]))
    y_min = float(np.min(hist["y"]))
    y_max = float(np.max(hist["y"]))
    print(f"Validation: x range = [{x_min:.4f}, {x_max:.4f}] m")
    print(f"Validation: y range = [{y_min:.4f}, {y_max:.4f}] m")
    print("Validation: disturbance stays within the restoring basin." if abs(x_max) < 0.012 and abs(x_min) < 0.012 else "Warning: disturbance is large; consider reducing disturbance_force.")
    print("Validation: hover height remains stable." if y_min > cfg["floor_y"] + 0.005 else "Warning: hover is too close to the floor.")

    writer = animation.FFMpegWriter(
        fps=cfg["fps"],
        bitrate=cfg["bitrate"],
        extra_args=["-pix_fmt", "yuv420p", "-movflags", "faststart"]
    )
    ani.save(cfg["video_path"], writer=writer, dpi=100)
    plt.close(fig)
    return cfg["video_path"]

# -------------------------------------------------------------------
# Run
# -------------------------------------------------------------------
np.random.seed(7)

print("Simulating stable jet dynamics...")
history = simulate_ball(CONFIG)

print("Rendering video...")
video_path = render_video(history, CONFIG)

if os.path.exists(video_path):
    display(Video(video_path, embed=True, width=360))
    files.download(video_path)
else:
    print("Video render failed.")


INFO:matplotlib.animation:Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
INFO:matplotlib.animation:MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 900x1600 -pix_fmt rgba -framerate 30 -loglevel error -i pipe: -vcodec h264 -b 900k -pix_fmt yuv420p -movflags faststart -y bernoulli_ping_pong_stability.mp4


Simulating stable jet dynamics...
Rendering video...
Validation: x range = [-0.0073, 0.0073] m
Validation: y range = [0.1211, 0.1441] m
Validation: disturbance stays within the restoring basin.
Validation: hover height remains stable.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>